In [ ]:
# LiitLLM — evaluation
#
# The headline question: does filtering a corpus FOR code-switching produce a
# model that code-switches? Both arms are scored with the same scorer that
# built the corpora, on identical prompts.

In [ ]:
import torch, sys
assert torch.cuda.is_available(), "no GPU — set the accelerator to T4 x2"
cap = torch.cuda.get_device_capability()
name = torch.cuda.get_device_name(0)
print(f"{name}  sm_{cap[0]}{cap[1]}")
assert cap >= (7, 0), (
    f"{name} is sm_{cap[0]}{cap[1]}; Kaggle's PyTorch needs sm_70+. "
    "Set --accelerator NvidiaTeslaT4 (the default P100 will not work)."
)

In [ ]:
import glob
from pathlib import Path

def find_one(slug, filename, kind):
    """Locate a mounted source by slug. `kind` is 'datasets' or 'notebooks'.

    Kaggle mounts sources at /kaggle/input/<kind>/<owner>/<slug>/..., NOT at the
    flat /kaggle/input/<slug>/ that most examples show — a glob written for the
    flat layout silently matches nothing.

    `kind` is not optional, and that is the point. Every kernel output carries a
    full copy of the repo, so a slug-anchored search across all of /kaggle/input
    finds BOTH the repo dataset and the repo copy embedded in the previous part's
    output. Scoping the search to where the thing legitimately lives —
    the repo always in 'datasets', corpora and checkpoints always in 'notebooks'
    — makes the ambiguity impossible instead of merely detected.
    """
    hits = glob.glob(f"/kaggle/input/{kind}/**/{slug}/**/{filename}", recursive=True)
    assert len(hits) == 1, (
        f"expected exactly 1 {filename} under {kind}/{slug}, found {hits}.\n"
        f"Sources actually mounted: {sorted(glob.glob('/kaggle/input/*/*/*'))}"
    )
    return Path(hits[0])

In [ ]:
import shutil, os, sys

REPO_SLUG = "liitllm-repo"   # dataset holding this repo
PREP_SLUG = "00-prep"        # kernel whose OUTPUT holds the corpora + tokenizer

repo_src = find_one(REPO_SLUG, "pyproject.toml", "datasets").parent
REPO = Path("/kaggle/working/liitllm-repo")
if REPO.exists():
    shutil.rmtree(REPO)
shutil.copytree(repo_src, REPO)
sys.path.insert(0, str(REPO))
os.chdir(REPO)
print(f"repo: {repo_src} -> {REPO}")

In [ ]:
# Final part of each run holds its final checkpoint. Both seeds per arm are
# passed so the verdict can compare the between-arm gap against the
# within-arm spread — which is the entire reason for running a second seed.
BASELINE = ['baseline-part4', 'baseline2-part4']
ABLATION = ['ablation-part4', 'ablation2-part4']

In [ ]:
from liitllm.evaluate import compare
import glob
def ckpt(slug):
    hits = glob.glob(f'/kaggle/input/notebooks/**/{slug}/**/ckpt.pt', recursive=True)
    assert len(hits) == 1, f'expected 1 ckpt.pt under {slug}, found {hits}'
    return hits[0]
# The tokenizer must be the one BOTH arms trained against, so it comes from
# the prep kernel's output — not from either checkpoint's neighbourhood.
tok = str(find_one(PREP_SLUG, 'tokenizer.json', 'notebooks'))
compare([ckpt(s) for s in BASELINE], [ckpt(s) for s in ABLATION],
        tok, out_dir='/kaggle/working/results')